<a href="https://colab.research.google.com/github/talhanoor23/algorithmic-trading/blob/main/Tradingbot_using_NeuralNetworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

import json

# Notebook filename
notebook ="/content/drive/MyDrive/Algo_Trading/Pandas_ta/Tradingbot_using_NeuralNetworks.ipynb"   # <-- Change this

with open(notebook, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove widget metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

# Also remove widget outputs
for cell in nb.get("cells", []):
    if "outputs" in cell:
        for output in cell["outputs"]:
            if output.get("output_type") == "display_data":
                output.get("data", {}).pop("application/vnd.jupyter.widget-view+json", None)

with open(notebook, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

print("Done! Widget metadata removed.")

Done! Widget metadata removed.


In [ ]:
import tensorflow as tf

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential

In [ ]:
apikey = '************************'
symbol = "BTC/USD"
interval = "5min"
order = "asc"
start_date = "2020-06-03"
end_date = "2025-06-21"

In [ ]:
# url = f"https://api.twelvedata.com/time_series?symbol=BTC/USD&interval=5min&order=asc&apikey={apikey}&limit=1000"
url = f"https://api.twelvedata.com/time_series?symbol={symbol}&interval={interval}&start_date={start_date}&end_date={end_date}&apikey={apikey}"

In [ ]:
import requests
data = requests.get(url).json()
print(data)

{'meta': {'symbol': 'BTC/USD', 'interval': '5min', 'currency_base': 'Bitcoin', 'currency_quote': 'US Dollar', 'exchange': 'Coinbase Pro', 'type': 'Digital Currency'}, 'values': [{'datetime': '2025-06-21 00:00:00', 'open': '103317.79', 'high': '103317.8', 'low': '103229.39', 'close': '103263.97'}, {'datetime': '2025-06-20 23:55:00', 'open': '103305.99', 'high': '103326.29', 'low': '103286.7', 'close': '103317.8'}, {'datetime': '2025-06-20 23:50:00', 'open': '103284.6', 'high': '103316.35', 'low': '103275.14', 'close': '103306'}, {'datetime': '2025-06-20 23:45:00', 'open': '103319.09', 'high': '103359.99', 'low': '103245.19', 'close': '103284.58'}, {'datetime': '2025-06-20 23:40:00', 'open': '103299.5', 'high': '103323.24', 'low': '103245', 'close': '103319.09'}, {'datetime': '2025-06-20 23:35:00', 'open': '103349.13', 'high': '103349.13', 'low': '103261.01', 'close': '103299.5'}, {'datetime': '2025-06-20 23:30:00', 'open': '103245.4', 'high': '103354.79', 'low': '103240.75', 'close': '1

In [ ]:
if "values" in data:
    df = pd.DataFrame(data["values"])
    df['datetime'] = pd.to_datetime(df['datetime'])

    # ✅ Force ascending sort (earliest to latest)
    df = df.sort_values(by="datetime", ascending=True).reset_index(drop=True)

    print("✅ Data loaded:", len(df), "rows")
    print("First datetime:", df['datetime'].iloc[0])
    print("Last datetime :", df['datetime'].iloc[-1])
    print(df.head())
else:
    print("❌ Error:", data.get("message", "Unknown error"))

✅ Data loaded: 5000 rows
First datetime: 2025-06-03 15:25:00
Last datetime : 2025-06-21 00:00:00
             datetime       open       high        low      close
0 2025-06-03 15:25:00  106733.27  106840.68  106733.27  106805.45
1 2025-06-03 15:30:00  106805.45     106859     106641  106647.05
2 2025-06-03 15:35:00  106647.05  106834.68  106647.05  106777.83
3 2025-06-03 15:40:00  106777.74  106851.22  106756.98     106757
4 2025-06-03 15:45:00     106757  106785.21  106541.82  106541.82


In [ ]:
# import requests
# import pandas as pd

# def fetch_cc(symbol='BTC', limit=2000):
#     url = "https://min-api.cryptocompare.com/data/v2/histominute"
#     params = {
#         'fsym': symbol,
#         'tsym': 'USD',
#         'limit': limit,
#         'aggregate': 5  # ✅ 5-minute bars
#     }
#     r = requests.get(url, params=params)
#     data = r.json()['Data']['Data']
#     df = pd.DataFrame(data)
#     df['time'] = pd.to_datetime(df['time'], unit='s')
#     return df

# df = fetch_cc()
# print(f"✅ CryptoCompare data: {len(df)} rows")
# print(df.head())


In [ ]:
df

,datetime,open,high,low,close
0,2025-06-03 15:25:00,106733.27,106840.68,106733.27,106805.45
1,2025-06-03 15:30:00,106805.45,106859,106641,106647.05
2,2025-06-03 15:35:00,106647.05,106834.68,106647.05,106777.83
3,2025-06-03 15:40:00,106777.74,106851.22,106756.98,106757
4,2025-06-03 15:45:00,106757,106785.21,106541.82,106541.82
...,...,...,...,...,...
4995,2025-06-20 23:40:00,103299.5,103323.24,103245,103319.09
4996,2025-06-20 23:45:00,103319.09,103359.99,103245.19,103284.58
4997,2025-06-20 23:50:00,103284.6,103316.35,103275.14,103306
4998,2025-06-20 23:55:00,103305.99,103326.29,103286.7,103317.8


In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(df['close'].values.reshape(-1,1))

In [ ]:
time_interval_to_train = 24
prediction_interval = 12
x_train, y_train = [], []

In [ ]:
# for i in range(time_interval_to_train, len(scaled_data) - prediction_interval + 1):
#     x_train.append(scaled_data[i-time_interval_to_train:i, 0])
#     y_train.append(scaled_data[i+prediction_interval-1:i+prediction_interval, 0])

In [ ]:
for i in range(time_interval_to_train, len(scaled_data) - prediction_interval):
    x_train.append(scaled_data[i-time_interval_to_train:i, 0])
    y_train.append(scaled_data[i+prediction_interval, 0])

x_train, y_train = np.array(x_train), np.array(y_train)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

In [ ]:
x_train.shape

(4964, 24, 1)

In [ ]:
model = Sequential()
model.add(LSTM(128, return_sequences=True, input_shape=(x_train.shape[1], 1), activation='relu'))
model.add(Dropout(0.4))
model.add(LSTM(64, return_sequences=True, activation='relu'))
model.add(Dropout(0.3))
model.add(LSTM(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation = 'sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(x_train, y_train, epochs=10, batch_size=64)

Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 13s 94ms/step - accuracy: 3.4796e-04 - loss: 0.0231
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 82ms/step - accuracy: 0.0011 - loss: 0.0042
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 101ms/step - accuracy: 2.5571e-04 - loss: 0.0032
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - accuracy: 4.6878e-04 - loss: 0.0031
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 78ms/step - accuracy: 5.7767e-04 - loss: 0.0028
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 97ms/step - accuracy: 0.0012 - loss: 0.0027
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 78ms/step - accuracy: 4.5121e-04 - loss: 0.0027
Epoch 8/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 99ms/step - accuracy: 8.7577e-04 - loss: 0.0026
Epoch 9/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 79ms/step - accuracy: 6.5043e-05 - loss: 0.0024
Epoch 10/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 7s 96ms/step - accuracy: 4.8355e-04 - loss: 0.0025
